In [1]:
# use datacq kernel

import getpass
import hashlib
import json
import os
import random
import time
from pathlib import Path

import pandas as pd
import requests
from requests.structures import CaseInsensitiveDict

ROOT_DIR = Path.cwd().parent

CACHE = ROOT_DIR / "cache"
CACHE.mkdir(exist_ok=True)
FIXTURES = Path("fixtures")

DATA_DIR = ROOT_DIR / "data"
DATA_DIR.mkdir(exist_ok=True)

### credentials input


In [2]:
# Endpoints
TOKEN_URL = "https://id.twitch.tv/oauth2/token"
STREAMS_URL = "https://api.twitch.tv/helix/streams"
USERS_URL = "https://api.twitch.tv/helix/users"

# Credentials
CLIENT_ID = os.environ.get("TWITCH_CLIENT_ID")
CLIENT_SECRET = os.environ.get("TWITCH_CLIENT_SECRET")

if not CLIENT_ID:
    CLIENT_ID = getpass.getpass("Twitch Client ID: ")
if not CLIENT_SECRET:
    CLIENT_SECRET = getpass.getpass("Twitch Client Secret: ")

In [3]:
OFFLINE = os.environ.get("OFFLINE") == "1"
POLITE_DELAY = 0.0 if OFFLINE else 1.0   # seconds between live requests


def cache_key(url, params=None):
    """A short, stable name for one (url, params) pair.
    """
    payload = url + "|" + json.dumps(params or {}, sort_keys=True)
    return hashlib.sha256(payload.encode()).hexdigest()[:16]


class FixtureResponse:

    def __init__(self, record):
        self.url = record.get("final_url", record["url"])
        self.status_code = record["status_code"]
        self.headers = CaseInsensitiveDict(record["headers"])
        self._body = record["body"]

    def json(self):
        return self._body

    @property
    def links(self):
        header = self.headers.get("link")
        if not header:
            return {}
        return {l["rel"]: l for l in requests.utils.parse_header_links(header)}

    def raise_for_status(self):
        if self.status_code >= 400:
            raise requests.HTTPError(f"{self.status_code} for {self.url}")


def offline_get(url, params=None, **kwargs):
    path = FIXTURES / f"{cache_key(url, params)}.json"
    if not path.exists():
        raise RuntimeError(
            f"no fixture for {url} {params}")
    return FixtureResponse(json.loads(path.read_text()))

if OFFLINE:
    requests.get = offline_get
    print("OFFLINE:", len(list(FIXTURES.glob("*.json"))), "fixtures loaded")
else:
    print("live requests")

live requests


### token exchange

In [4]:
def get_oauth_token(cid, secret):
  payload = {
      "client_id": cid,
      "client_secret": secret,
      "grant_type": "client_credentials",
  }
  r = requests.post(TOKEN_URL, data=payload, timeout=10)
  r.raise_for_status()
  return r.json()["access_token"]


TOKEN = get_oauth_token(CLIENT_ID, CLIENT_SECRET) if not OFFLINE else "mock_token"

HEADERS = {"Client-Id": CLIENT_ID, "Authorization": f"Bearer {TOKEN}"}

In [5]:
def http_get(url, params=None, headers=HEADERS, attempts=5):
    for k in range(attempts):
        try:
            r = requests.get(url, params=params, headers=headers,
                             timeout=10)
        except requests.RequestException as exc:   # timeout, DNS, reset
            status, retry_after = repr(exc), 0
        else:
            # 200 -> return the parsed body
            if (r.status_code == 200):
                return r.json()

            # status that will not change -> raise_for_status()
            if r.status_code not in (429, 500, 502, 503, 504):
                r.raise_for_status()


            # 3. otherwise remember the status, and read Ratelimit-Reset
            status = r.status_code
            retry_after = 0
            if "Ratelimit-Reset" in r.headers:
                try:    # this is a timestamp : we can derive wait time
                    reset_epoch = int(r.headers["Ratelimit-Reset"])
                    retry_after = max(0, reset_epoch - int(time.time()))
                except (ValueError, TypeError):
                    retry_after = 0


        if k + 1 < attempts:

            if retry_after > 0:
                wait = retry_after
            else:
                wait = random.uniform(0, 2**k)

            print(f"  {status} on attempt {k + 1}, sleeping {wait:.1f}s")
            time.sleep(wait)
            
    raise RuntimeError(f"gave up after {attempts} attempts: {url}")


In [6]:
requests_made = 0


def cached_get(url, params=None):
    path = CACHE / f"{cache_key(url, params)}.json"
    if path.exists():
        return json.loads(path.read_text())

    global requests_made
    requests_made += 1
    body = http_get(url, params)
    path.write_text(json.dumps(body))
    time.sleep(POLITE_DELAY)
    return body

### fetch streams loop

Twitch uses cursor-based pagination ;
[API guide](https://dev.twitch.tv/docs/api/guide/)

Be careful to move data to the root/data folder, which is in the .gitignore file, to avoid uploading all data to github.

In [7]:

TARGET_RECORDS = 10_000
PAGE_LIMIT = 100



streams, page, cursor = [], 1, None


while len(streams) < TARGET_RECORDS:
  params = {"language": "en", "first": PAGE_LIMIT}
  if cursor:
    params["after"] = cursor

  data = cached_get(STREAMS_URL, params=params)
  batch = data.get("data", [])

  if not batch:
    break

  streams.extend(batch)
  print(
      f"Page {page}: collected {len(batch)} streams (running total:"
      f" {len(streams)})"
  )

  cursor = data.get("pagination", {}).get("cursor")
  if not cursor:
    break

  page += 1

print(f"\nTotal live streams collected: {len(streams)}")

Page 1: collected 100 streams (running total: 100)
Page 2: collected 99 streams (running total: 199)
Page 3: collected 99 streams (running total: 298)
Page 4: collected 98 streams (running total: 396)
Page 5: collected 100 streams (running total: 496)
Page 6: collected 100 streams (running total: 596)
Page 7: collected 100 streams (running total: 696)
Page 8: collected 100 streams (running total: 796)
Page 9: collected 99 streams (running total: 895)
Page 10: collected 100 streams (running total: 995)
Page 11: collected 97 streams (running total: 1092)
Page 12: collected 99 streams (running total: 1191)
Page 13: collected 100 streams (running total: 1291)
Page 14: collected 98 streams (running total: 1389)
Page 15: collected 100 streams (running total: 1489)
Page 16: collected 98 streams (running total: 1587)
Page 17: collected 98 streams (running total: 1685)
Page 18: collected 97 streams (running total: 1782)
Page 19: collected 99 streams (running total: 1881)
Page 20: collected 98 s

### fetch broadcasters loop

Here we get user metadata in batches of 100 to avoid hitting rate limit

fix: don't store uneccessary user personal info


In [12]:
# Extract unique user IDs
user_ids = list(dict.fromkeys(s["user_id"] for s in streams))
broadcasters = []


BATCH_SIZE = 100
for i in range(0, len(user_ids), BATCH_SIZE):
  chunk = user_ids[i : i + BATCH_SIZE]
  # Pass list of tuples to handle multiple 'id' query parameters
  params = [("id", uid) for uid in chunk]

  data = cached_get(USERS_URL, params=params)
  broadcasters.extend(data.get("data", []))

print(f"Broadcaster profiles collected: {len(broadcasters)}")

Broadcaster profiles collected: 8684


### storing the dataset and the log

In [ ]:
from datetime import datetime, timezone

df_streams = pd.json_normalize(streams)
df_broadcasters = pd.json_normalize(broadcasters)

streams_path = DATA_DIR / "streams.parquet"
broadcasters_path = DATA_DIR / "broadcasters.parquet"

df_streams.to_parquet(streams_path, index=False)
df_broadcasters.to_parquet(broadcasters_path, index=False)

# Acquisition log in root
log = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "api_version": "Twitch Helix",
    "endpoints": [STREAMS_URL, USERS_URL],
    "parameters": {
        "language": "en",
        "target_count": TARGET_RECORDS,
        "batch_size": PAGE_LIMIT,
    },
    "collected_counts": {
        "streams": len(df_streams),
        "broadcasters": len(df_broadcasters),
    },
    "shapes": {
        "streams_parquet": list(df_streams.shape),
        "broadcasters_parquet": list(df_broadcasters.shape),
    },
    "storage_files": [str(streams_path), str(broadcasters_path)],
}

Path( ROOT_DIR /"acquisition_log.json").write_text(json.dumps(log, indent=2))
print("Acquisition log written to root: acquisition_log.json")

Acquisition log written to root: acquisition_log.json
